<a href="https://colab.research.google.com/github/HermesRoe/Food_prices_AI/blob/main/food_prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd

# 1. PASTE YOUR ACTUAL RAW LINK HERE
csv_url = 'https://raw.githubusercontent.com/HermesRoe/Food_prices_AI/refs/heads/main/wfp_food_prices_ind.csv'

# 2. Try loading again
try:
    df = pd.read_csv(csv_url, low_memory=False)
    print("Step 1: Data loaded successfully from GitHub!")
    print(df.head(2)) # Show just 2 rows to confirm
except Exception as e:
    print("Error: Still can't find the file. Check the link!")
    print(e)

Step 1: Data loaded successfully from GitHub!
         date      admin1      admin2            market   latitude  longitude  \
0       #date  #adm1+name  #adm2+name  #loc+market+name   #geo+lat   #geo+lon   
1  1994-01-15       Delhi       Delhi             Delhi  28.666667  77.216667   

             category   commodity        unit         priceflag  \
0          #item+type  #item+name  #item+unit  #item+price+flag   
1  cereals and tubers        Rice          KG            actual   

          pricetype   currency   price    usdprice  
0  #item+price+type  #currency  #value  #value+usd  
1            Retail        INR     8.0      0.2545  


In [17]:
# 1. Remove the HXL tags (Row 0)
df = df.drop(df.index[0])

# 2. Filter and standardize
india_df = df[df['admin1'] == 'Delhi'].copy()

# 3. Convert types
india_df['price'] = pd.to_numeric(india_df['price'], errors='coerce')
india_df['date'] = pd.to_datetime(india_df['date'])
india_df = india_df[india_df['unit'] == 'KG']

# 4. Create Year/Month columns
india_df['year'] = india_df['date'].dt.year
india_df['month'] = india_df['date'].dt.month

print("Step 2: Data cleaned and 'india_df' created!")

Step 2: Data cleaned and 'india_df' created!


In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Prepare data
X = india_df[['year', 'month']]
y = india_df['price']

# Train Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X.values, y)

print("Step 3: AI Model Trained!")

Step 3: AI Model Trained!


In [19]:
# 1. Remove the HXL tags (Row 0)
df = df.drop(df.index[0])

# 2. Filter for India (if not already filtered) and standardize
india_df = df[df['admin1'] == 'Delhi'].copy() # Focus on Delhi for best accuracy

# 3. Clean Units and Types
india_df['price'] = pd.to_numeric(india_df['price'], errors='coerce')
india_df['date'] = pd.to_datetime(india_df['date'])
india_df = india_df[india_df['unit'] == 'KG'] # Essential for that 0.13 error margin!

# 4. Create Time Features
india_df['year'] = india_df['date'].dt.year
india_df['month'] = india_df['date'].dt.month

print("Cleanup successful! Data is standard and ready.")
india_df.head()

Cleanup successful! Data is standard and ready.


,date,admin1,admin2,market,latitude,longitude,category,commodity,unit,priceflag,pricetype,currency,price,usdprice,year,month
2,1994-01-15,Delhi,Delhi,Delhi,28.666667,77.216667,cereals and tubers,Wheat,KG,actual,Retail,INR,5.0,0.159,1994,1
3,1994-01-15,Delhi,Delhi,Delhi,28.666667,77.216667,miscellaneous food,Sugar,KG,actual,Retail,INR,13.5,0.4294,1994,1
4,1994-01-15,Delhi,Delhi,Delhi,28.666667,77.216667,oil and fats,Oil (mustard),KG,actual,Retail,INR,31.0,0.986,1994,1
34,1994-02-15,Delhi,Delhi,Delhi,28.666667,77.216667,cereals and tubers,Rice,KG,actual,Retail,INR,8.0,0.2545,1994,2
35,1994-02-15,Delhi,Delhi,Delhi,28.666667,77.216667,cereals and tubers,Wheat,KG,actual,Retail,INR,5.2,0.1654,1994,2


In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# 1. Train the AI on the cleaned 'india_df'
# We focus on Rice for this demo
rice_df = india_df[india_df['commodity'] == 'Rice'].dropna(subset=['year', 'month', 'price'])
X = rice_df[['year', 'month']]
y = rice_df['price']

final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X.values, y)

# 2. Build the Platform Interface
year_slider = widgets.IntText(value=2026, description='Year:')
month_slider = widgets.IntSlider(value=1, min=1, max=12, description='Month:')
btn = widgets.Button(description='Predict Market Price', button_style='success')
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        # Get the prediction
        prediction = final_model.predict([[year_slider.value, month_slider.value]])[0]

        print(f"--- AI FOOD PRICE PREDICTION PLATFORM ---")
        print(f"Commodity: Rice | Market: Delhi")
        print(f"Predicted Price: ₹{prediction:.2f} per KG")
        print(f"Status: Stability Analyzed")

        # Show a quick trend graph for that year
        future_months = [[year_slider.value, m] for m in range(1, 13)]
        future_prices = final_model.predict(future_months)

        plt.figure(figsize=(8,3))
        plt.plot(range(1, 13), future_prices, 'g--o')
        plt.xticks(range(1, 13), ['J','F','M','A','M','J','J','A','S','O','N','D'])
        plt.title(f"Price Trend Forecast for {year_slider.value}")
        plt.show()

btn.on_click(on_click)
display(year_slider, month_slider, btn, out)

IntText(value=2026, description='Year:')

IntSlider(value=1, description='Month:', max=12, min=1)

Button(button_style='success', description='Predict Market Price', style=ButtonStyle())

Output()